In [7]:
from openai import OpenAI

BASE_URL = "http://localhost:11434/v1"
MODELL = "llama3.2"

client = OpenAI(base_url=BASE_URL, api_key="ollama")

In [8]:
antwort = client.chat.completions.create(
    model=MODELL,
    messages=[
        {"role": "system", "content": "Du bist der Einkaufsassistent des Hotel Seeblick. Antworte kurz auf Deutsch."},
        {"role": "user", "content": "Was ist ein Mindestbestand?"},
    ],
)
print(antwort.choices[0].message.content)

Im Hotel Seeblick beziehungsweise im Allgemeinen ist ein Mindestbestand die Menge an Produkten oder Ware, die ein Unternehmen oder Geschäft in einem bestimmten Zeitraum halten muss. Dies kann aufgrund verschiedener Faktoren wie Lieferverzugsfristen, Produktionsplanung oder Marktbedürfnisse erforderlich sein.


In [9]:
from typing import Literal

from pydantic import BaseModel, Field


class Bedarf(BaseModel):
    artikel: str = Field(description="Name des Artikels, z. B. Mehl")
    menge: int = Field(gt=0, description="Bestellmenge als ganze Zahl")
    einheit: Literal["kg", "Liter", "Stück", "Packung"] = Field(
        description="Einheit; bei zählbaren Dingen wie Eiern immer Stück"
    )


class Bestellanfrage(BaseModel):
    abteilung: Literal["Küche", "Housekeeping", "Rezeption"] = Field(
        description="Die Abteilung, die die Ware braucht – nicht der Empfänger der Nachricht"
    )
    dringend: bool
    bedarfe: list[Bedarf]

In [10]:
nachricht = """Hallo Einkauf,
die Küche braucht bis Freitag dringend 30 kg Mehl, 200 Eier und 5 Liter Olivenöl.
Grüße, Chefkoch"""

antwort = client.chat.completions.parse(
    model=MODELL,
    messages=[
        {"role": "system", "content": "Du bist der Einkaufsassistent des Hotel Seeblick." 
            "Extrahiere aus der Nachricht eine Bestellanfrage. Die Abteilung ist der Absender, nicht der Einkauf."},
        {"role": "user", "content": nachricht},
    ],
    response_format=Bestellanfrage,
)

anfrage = antwort.choices[0].message.parsed
print(anfrage)
print()
for bedarf in anfrage.bedarfe:
    print(f"{bedarf.menge} {bedarf.einheit} {bedarf.artikel}")

abteilung='Küche' dringend=True bedarfe=[Bedarf(artikel='Mehl', menge=30, einheit='kg'), Bedarf(artikel='Eier', menge=200, einheit='Stück'), Bedarf(artikel='Olivenöl', menge=5, einheit='Liter')]

30 kg Mehl
200 Stück Eier
5 Liter Olivenöl


In [20]:
# ===== TOOLS: Werkzeuge für das Modell =====
import json

import httpx

API = "http://127.0.0.1:8000"     # Adresse deiner MiniERP-API
MANDANT = {"X-Mandant": "1"}      # Header: 1 = Hotel Seeblick


# Echte Funktion 1: fragt die API nach Artikeln unter Mindestbestand
def artikel_unter_mindestbestand() -> list[dict]:
    return httpx.get(f"{API}/einkauf/artikel/unter-mindestbestand", headers=MANDANT).json()


# Echte Funktion 2: fragt die API nach Lieferantenangeboten für einen Artikel
def angebote_fuer_artikel(artikel_id: int) -> list[dict]:
    return httpx.get(f"{API}/einkauf/artikel/{artikel_id}/angebote", headers=MANDANT).json()


# Gebrauchsanweisung für das Modell: Name, WANN benutzen, welche Parameter.
# Das Modell sieht nur diesen Text, nie die Funktionen oben.
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "artikel_unter_mindestbestand",
            "description": (
                "Benutze dieses Tool, wenn gefragt wird, was nachbestellt werden muss "
                "oder welche Artikel knapp sind. Braucht keine Parameter. "
                "Liefert alle Artikel unter Mindestbestand mit id, name, bestand und mindestbestand."
            ),
            "parameters": {"type": "object", "properties": {}},
        },
    },
    {
        "type": "function",
        "function": {
            "name": "angebote_fuer_artikel",
            "description": (
                "Liefert die Lieferantenangebote für EINEN Artikel, günstigster zuerst. "
                "Nur benutzen, wenn die artikel_id aus einem vorherigen Tool-Ergebnis bekannt ist. "
                "Niemals eine artikel_id erfinden."
            ),
            "parameters": {
                "type": "object",
                "properties": {"artikel_id": {"type": "integer", "description": "ID aus einem vorherigen Tool-Ergebnis"}},
                "required": ["artikel_id"],
            },
        },
    },
]

# Zuordnung: Tool-Name → echte Python-Funktion
FUNKTIONEN = {
    "artikel_unter_mindestbestand": artikel_unter_mindestbestand,
    "angebote_fuer_artikel": angebote_fuer_artikel,
}

print("Tools geladen.")


Tools geladen.


In [21]:
# ===== FRAGE: Modell bekommt Frage + Tools und entscheidet =====

# Systemprompt in Teilen – Python fügt die Texte in Klammern zusammen
SYSTEM = (
    "Du bist der Einkaufsassistent des Hotel Seeblick. "
    "Beantworte Fragen nur mit Daten aus den Tools. "
    "Für die Frage, was nachbestellt werden muss, nutze artikel_unter_mindestbestand. "
    "Erfinde niemals IDs oder Daten. Antworte kurz auf Deutsch."
)

# Gesprächsverlauf – wird bei jedem Ausführen dieser Zelle neu angelegt
messages = [
    {"role": "system", "content": SYSTEM},
    {"role": "user", "content": "Welche Artikel müssen wir nachbestellen?"},
]

# Anfrage ans Modell, mit Werkzeugliste
antwort = client.chat.completions.create(
    model=MODELL,
    messages=messages,
    tools=TOOLS,
)

modell_nachricht = antwort.choices[0].message
print("Text:", modell_nachricht.content)            # direkte Antwort (oder leer)
print("Tool-Aufrufe:", modell_nachricht.tool_calls) # angeforderte Tools (oder None)

Text: 
Tool-Aufrufe: [ChatCompletionMessageFunctionToolCall(id='call_9i6v1hr8', function=Function(arguments='{}', name='artikel_unter_mindestbestand'), type='function', index=0)]


In [22]:
# 1. Die Tool-Anfrage des Modells in den Verlauf aufnehmen.
#    Ohne sie wüsste das Modell beim nächsten Aufruf nicht,
#    dass es ein Tool angefordert hat (es hat kein Gedächtnis).
messages.append(modell_nachricht)

# 2. Alle angeforderten Tools nacheinander ausführen
#    (das Modell kann auch mehrere auf einmal anfordern)
for aufruf in modell_nachricht.tool_calls:
    # Name des gewünschten Tools, z. B. "artikel_unter_mindestbestand"
    name = aufruf.function.name

    # Passende Python-Funktion über das Wörterbuch aus Schritt 4 finden
    funktion = FUNKTIONEN[name]

    # Argumente kommen als JSON-Text, z. B. '{"artikel_id": 4}'
    # → in ein Python-Dictionary umwandeln. Leer? Dann "{}" nehmen.
    argumente = json.loads(aufruf.function.arguments or "{}")

    # Funktion ausführen. ** setzt das Dictionary als Parameter ein:
    # funktion(**{"artikel_id": 4})  ist dasselbe wie  funktion(artikel_id=4)
    ergebnis = funktion(**argumente)
    print("Tool ausgeführt:", name, argumente)

    # 3. Ergebnis als Nachricht mit der Rolle "tool" anhängen.
    #    tool_call_id verbindet das Ergebnis mit der Anfrage von oben.
    #    Inhalt muss Text sein → Python-Liste in JSON-Text umwandeln.
    messages.append({
        "role": "tool",
        "tool_call_id": aufruf.id,
        "content": json.dumps(ergebnis, ensure_ascii=False),  # ensure_ascii=False: Umlaute bleiben lesbar
    })

# 4. Zweiter Aufruf: Jetzt hat das Modell die echten Daten im Verlauf
#    und kann eine richtige Antwort formulieren.
endantwort = client.chat.completions.create(
    model=MODELL,
    messages=messages,
    tools=TOOLS,
)

print()
print("Antwort:", endantwort.choices[0].message.content)


Tool ausgeführt: artikel_unter_mindestbestand {}

Antwort: Wir müssen die folgenden Artikel nachbestellen:

- 40 kg Weizenmehl (Artikelnummer: K-003)
- 10 Stück Rinderfilet (Artikelnummer: K-004)
- 300 Eier (Artikelnummer: K-006)
- 40 Handtücher (Artikelnummer: H-002)
- 30 Liter Allzweckreiniger (Artikelnummer: H-003)
